In [2]:
import pandas as pd 
df=pd.read_csv(r"data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())
print(df.info())
print(df['Churn'].value_counts())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
df_c=df.drop('customerID',axis=1)
df_c['TotalCharges']=pd.to_numeric(df_c['TotalCharges'],errors='coerce').fillna(0)

In [6]:
from sklearn.preprocessing import OneHotEncoder  
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

y=df_c['Churn'].map({'Yes':1,"No":0})
x=df_c.drop('Churn', axis=1)
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

all_columns=['PaymentMethod','PaperlessBilling','Contract','StreamingMovies','StreamingTV',
             'TechSupport','DeviceProtection','OnlineBackup','OnlineSecurity','InternetService','MultipleLines',
             'PhoneService','Dependents','Partner','gender']
preprocessor=ColumnTransformer(transformers=[('cat',OneHotEncoder(sparse_output=False,dtype=int),all_columns)],remainder='passthrough')
xtrain_decoded=preprocessor.fit_transform(x_train)
xtest_decoded=preprocessor.transform(x_test)
new_column=preprocessor.get_feature_names_out()
df_train_final=pd.DataFrame(xtrain_decoded,columns=new_column)
df_test_final=pd.DataFrame(xtest_decoded,columns=new_column)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
model1=LogisticRegression()
model1.fit(df_train_final,y_train)
predict1=model1.predict(df_test_final)
print('classification report: ',classification_report(y_test,predict1))
print('confusion_matrix: ',confusion_matrix(y_test,predict1))

/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


classification report:                precision    recall  f1-score   support

           0       0.86      0.91      0.88      1036
           1       0.70      0.57      0.63       373

    accuracy                           0.82      1409
   macro avg       0.78      0.74      0.76      1409
weighted avg       0.81      0.82      0.82      1409

confusion_matrix:  [[944  92]
 [159 214]]


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix

model=RandomForestClassifier(random_state=42)
model.fit(df_train_final,y_train)
predict=model.predict(df_test_final)

print('classification report: ',classification_report(y_test,predict))
print('confusion_matrix: ',confusion_matrix(y_test,predict))

classification report:                precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.66      0.47      0.55       373

    accuracy                           0.80      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.80      0.78      1409

confusion_matrix:  [[946  90]
 [197 176]]


In [48]:
coefficient=(model1.coef_).flatten()
data_columns=df_train_final.columns

df_cof=pd.DataFrame({'columns: ':data_columns,
                     'coefficients:':coefficient}).sort_values(by="coefficients:",ascending=False)

print(df_cof.head(5))
print(df_cof.tail(5))

                              columns:   coefficients:
6          cat__Contract_Month-to-month       0.285312
2   cat__PaymentMethod_Electronic check       0.280888
28     cat__InternetService_Fiber optic       0.263602
24               cat__OnlineSecurity_No       0.249494
15                  cat__TechSupport_No       0.244934
                   columns:   coefficients:
8     cat__Contract_Two year      -0.254186
4   cat__PaperlessBilling_No      -0.268738
17      cat__TechSupport_Yes      -0.274827
26   cat__OnlineSecurity_Yes      -0.279387
27  cat__InternetService_DSL      -0.293495


In [52]:
prob=model1.predict_proba(df_test_final)
print(prob[:,1])

[0.69022734 0.05641684 0.00640887 ... 0.05185821 0.01641625 0.45902298]


<class 'AttributeError'>: 'NoneType' object has no attribute 'head'

In [49]:
import joblib
joblib.dump(preprocessor,'preprocessor.pkl')
joblib.dump(model1,'model1.pkl')

['model1.pkl']